# 03 - Auto-ETS, Theta Method e Combinacao de Previsoes

Este notebook cobre a **selecao automatica** de modelos ETS, o **metodo Theta** e a **combinacao de previsoes** usando **chronobox**.

**Conteudo:**
1. Selecao automatica de modelo ETS (AIC/BIC)
2. Theta method como caso especial de SES com drift
3. Combinacao (ensemble) de previsoes
4. Comparacao: Auto-ETS vs Theta vs ARIMA
5. Exercicios

## 1. Selecao Automatica de Modelo ETS

### Criterios de Informacao

A funcao `auto_ets` avalia multiplos modelos e seleciona o melhor por criterio de informacao:

- **AIC** (Akaike): $AIC = -2\ln(L) + 2k$
- **BIC** (Bayesian): $BIC = -2\ln(L) + k\ln(n)$
- **AICc** (corrigido): $AICc = AIC + \frac{2k(k+1)}{n-k-1}$

onde $L$ e a verossimilhanca, $k$ o numero de parametros e $n$ o tamanho da amostra.

O AICc e preferido para amostras pequenas, pois penaliza mais a complexidade.

### Processo de selecao

1. Enumerar modelos candidatos (ate 30)
2. Ajustar cada modelo por maxima verossimilhanca
3. Calcular criterio de informacao
4. Selecionar o modelo com menor criterio
5. Verificar estabilidade e convergencia

## 2. Setup e Carregamento dos Dados

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys

from chronobox import ETS, HoltWinters, ThetaMethod, ARIMA
from chronobox.selection.auto_ets import auto_ets

np.random.seed(42)

DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data')

# Carregar datasets
airline = pd.read_csv(os.path.join(DATA_DIR, 'airline.csv'), parse_dates=['date'])
airline.set_index('date', inplace=True)
y_airline = airline['passengers'].values

ets_synth = pd.read_csv(os.path.join(DATA_DIR, 'ets_synthetic.csv'), parse_dates=['date'])
ets_synth.set_index('date', inplace=True)
y_synth = ets_synth['value'].values

print(f'Airline: {len(y_airline)} obs')
print(f'Sintetico: {len(y_synth)} obs')
print('\nModulos carregados: ETS, HoltWinters, ThetaMethod, ARIMA, auto_ets')

## 3. Auto-ETS: Selecao Automatica

In [ ]:
# Auto-ETS no dataset airline
result = auto_ets(y_airline, seasonal_period=12, information_criterion='aicc', verbose=True)

print('\n' + '=' * 60)
print(result.summary())
print(f'\nModelo selecionado: ETS({result.best_spec[0]},{result.best_spec[1]},{result.best_spec[2]})')
print(f'Modelos avaliados: {result.n_models_tried}')
print(f'Modelos com falha: {result.n_models_failed}')

In [ ]:
# Top 5 modelos por AICc
all_res = result.all_results
# Ordenar por AICc
sorted_res = sorted(all_res, key=lambda x: x[1].aicc if hasattr(x[1], 'aicc') else x[1].aic)

print(f'{"Rank":<6} {"Modelo":<20} {"AICc":>10} {"AIC":>10} {"BIC":>10}')
print('-' * 60)
for i, (spec, res) in enumerate(sorted_res[:10]):
    nome = f'ETS({spec[0]},{spec[1]},{spec[2]})'
    aicc = res.aicc if hasattr(res, 'aicc') else res.aic
    print(f'{i+1:<6} {nome:<20} {aicc:>10.2f} {res.aic:>10.2f} {res.bic:>10.2f}')

print(f'\nDiferenca AICc entre 1o e 2o: {sorted_res[1][1].aicc - sorted_res[0][1].aicc:.2f}')
print('(Diferenca > 2 indica evidencia forte a favor do melhor modelo)')

In [ ]:
# Grafico 1: Ajuste do modelo auto-selecionado
best = result.best_model
spec = result.best_spec

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(y_airline, label='Observado', color='steelblue', linewidth=1.5)
ax.plot(best.fittedvalues, label=f'Auto-ETS: ({spec[0]},{spec[1]},{spec[2]})',
        color='darkorange', linestyle='--', linewidth=1.2)
ax.set_title(f'Auto-ETS Selecionado: ETS({spec[0]},{spec[1]},{spec[2]}) - Airline')
ax.set_xlabel('Observacao')
ax.set_ylabel('Passageiros')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Parametros estimados:')
print(f'  alpha = {best.alpha:.4f}')
if best.beta is not None:
    print(f'  beta  = {best.beta:.4f}')
if best.gamma is not None:
    print(f'  gamma = {best.gamma:.4f}')
if best.phi is not None:
    print(f'  phi   = {best.phi:.4f}')

## 4. Theta Method

### Teoria

O **metodo Theta** (Assimakopoulos & Nikolopoulos, 2000) decompoe a serie em duas "linhas theta":

1. $\theta = 0$: linha de regressao linear (tendencia)
2. $\theta = 2$: serie amplificada (curvatura dobrada)

A previsao combina:
- **SES** (suavizacao exponencial simples) aplicada a serie theta=2
- **Drift** da regressao linear

Hyndman & Billah (2003) mostraram que o Theta method e equivalente a SES com drift:

$$\hat{y}_{t+h|t} = l_t + \frac{b}{2} h$$

onde $l_t$ e o nivel do SES e $b$ e o slope da regressao.

> O Theta method venceu a competicao M3 (2000), superando metodos mais complexos!

In [ ]:
# Ajustar Theta Method no airline
theta = ThetaMethod(theta=2.0)
res_theta = theta.fit(y_airline)

print(res_theta.summary())
print(f'\nParametros:')
print(f'  alpha (SES): {res_theta.alpha:.4f}')
print(f'  slope (regressao): {res_theta.slope:.6f}')
print(f'  intercept: {res_theta.intercept:.4f}')
print(f'  drift: {res_theta.drift:.4f}')

In [ ]:
# Grafico 2: Theta Method - ajuste e previsao
fc_theta = res_theta.forecast(steps=24)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(range(len(y_airline)), y_airline, label='Observado', color='steelblue', linewidth=1.5)
ax.plot(range(len(y_airline)), res_theta.fittedvalues, label='Theta ajustado',
        color='darkorange', linestyle='--', alpha=0.7)

idx_fc = range(len(y_airline), len(y_airline) + 24)
ax.plot(idx_fc, fc_theta, label='Theta previsao', color='crimson', linewidth=2)
ax.axvline(len(y_airline) - 1, color='gray', linestyle=':', alpha=0.5)

ax.set_title('Theta Method (theta=2.0) - Airline Passengers')
ax.set_xlabel('Observacao')
ax.set_ylabel('Passageiros')
ax.legend()
plt.tight_layout()
plt.show()

print('Nota: O Theta method nao captura sazonalidade diretamente.')
print('Para dados sazonais, pode-se dessazonalizar antes e re-sazonalizar depois.')

## 5. Comparacao: Auto-ETS vs Theta vs ARIMA

Vamos comparar tres abordagens diferentes no dataset airline, dividindo em treino/teste:

In [ ]:
# Dividir em treino (primeiros 120 obs) e teste (ultimos 24 obs)
n_test = 24
y_train = y_airline[:-n_test]
y_test = y_airline[-n_test:]

print(f'Treino: {len(y_train)} obs')
print(f'Teste: {len(y_test)} obs')

# 1. Auto-ETS
res_auto = auto_ets(y_train, seasonal_period=12, information_criterion='aicc')
fc_auto = res_auto.forecast(steps=n_test)
spec_auto = res_auto.best_spec

# 2. Theta Method
theta_model = ThetaMethod(theta=2.0)
res_theta_tr = theta_model.fit(y_train)
fc_theta_tr = res_theta_tr.forecast(steps=n_test)

# 3. ARIMA (auto - usando ordens comuns para airline)
arima_model = ARIMA(order=(1, 1, 1), seasonal_order=(1, 1, 1, 12))
res_arima = arima_model.fit(y_train)
fc_arima_raw = res_arima.forecast(steps=n_test)
fc_arima = fc_arima_raw['forecast'] if isinstance(fc_arima_raw, dict) else fc_arima_raw

print(f'\nModelos ajustados:')
print(f'  Auto-ETS: ({spec_auto[0]},{spec_auto[1]},{spec_auto[2]})')
print(f'  Theta (theta=2.0)')
print(f'  SARIMA(1,1,1)(1,1,1,12)')

In [ ]:
# Metricas de erro no conjunto de teste
def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print(f'{"Modelo":<25} {"MAE":>10} {"RMSE":>10} {"MAPE(%)":>10}')
print('-' * 58)

for nome, fc in [('Auto-ETS', fc_auto), ('Theta', fc_theta_tr), ('SARIMA(1,1,1)', fc_arima)]:
    fc_arr = np.array(fc)[:n_test]
    print(f'{nome:<25} {mae(y_test, fc_arr):>10.2f} {rmse(y_test, fc_arr):>10.2f} {mape(y_test, fc_arr):>10.2f}')

In [ ]:
# Grafico 3: Comparacao de previsoes
fig, ax = plt.subplots(figsize=(14, 6))

# Treino
ax.plot(range(len(y_train)), y_train, label='Treino', color='steelblue', linewidth=1.5)

# Teste
idx_test = range(len(y_train), len(y_airline))
ax.plot(idx_test, y_test, label='Teste (real)', color='black', linewidth=2)

# Previsoes
ax.plot(idx_test, np.array(fc_auto)[:n_test], label=f'Auto-ETS ({spec_auto[0]},{spec_auto[1]},{spec_auto[2]})',
        color='darkorange', linewidth=1.5, linestyle='--')
ax.plot(idx_test, np.array(fc_theta_tr)[:n_test], label='Theta',
        color='green', linewidth=1.5, linestyle='-.')
ax.plot(idx_test, np.array(fc_arima)[:n_test], label='SARIMA(1,1,1)',
        color='crimson', linewidth=1.5, linestyle=':')

ax.axvline(len(y_train) - 1, color='gray', linestyle=':', alpha=0.5)
ax.set_title('Comparacao: Auto-ETS vs Theta vs SARIMA')
ax.set_xlabel('Observacao')
ax.set_ylabel('Passageiros')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

## 6. Combinacao de Previsoes (Ensemble)

A **combinacao de previsoes** frequentemente supera modelos individuais. A ideia e simples:

$$\hat{y}^{comb}_{t+h} = \sum_{i=1}^{M} w_i \hat{y}^{(i)}_{t+h}$$

Metodos de combinacao:
1. **Media simples** ($w_i = 1/M$): robusto, nao requer estimacao
2. **Media ponderada por inverso do MSE**: $w_i \propto 1/MSE_i$
3. **Regressao**: estimar pesos por OLS (risco de overfitting)

> Bates & Granger (1969) e Timmermann (2006): a media simples e dificil de superar!

In [ ]:
# Combinacao de previsoes
fc_auto_arr = np.array(fc_auto)[:n_test]
fc_theta_arr = np.array(fc_theta_tr)[:n_test]
fc_arima_arr = np.array(fc_arima)[:n_test]

# 1. Media simples
fc_mean = (fc_auto_arr + fc_theta_arr + fc_arima_arr) / 3

# 2. Media ponderada pelo inverso do RMSE (in-sample)
rmse_auto = np.sqrt(np.mean(res_auto.best_model.resid ** 2))
rmse_theta_is = np.sqrt(np.mean(res_theta_tr.resid ** 2))
rmse_arima_is = np.sqrt(np.mean(res_arima.residuals ** 2))

w = np.array([1/rmse_auto, 1/rmse_theta_is, 1/rmse_arima_is])
w = w / w.sum()

fc_weighted = w[0] * fc_auto_arr + w[1] * fc_theta_arr + w[2] * fc_arima_arr

# 3. Melhor modelo individual
best_individual = min(
    [('Auto-ETS', fc_auto_arr), ('Theta', fc_theta_arr), ('SARIMA', fc_arima_arr)],
    key=lambda x: rmse(y_test, x[1])
)

print(f'{"Combinacao":<30} {"MAE":>10} {"RMSE":>10} {"MAPE(%)":>10}')
print('-' * 63)
print(f'{"Media simples":<30} {mae(y_test, fc_mean):>10.2f} {rmse(y_test, fc_mean):>10.2f} {mape(y_test, fc_mean):>10.2f}')
print(f'{"Media ponderada (inv-RMSE)":<30} {mae(y_test, fc_weighted):>10.2f} {rmse(y_test, fc_weighted):>10.2f} {mape(y_test, fc_weighted):>10.2f}')
print(f'{"Melhor individual (" + best_individual[0] + ")":<30} {mae(y_test, best_individual[1]):>10.2f} {rmse(y_test, best_individual[1]):>10.2f} {mape(y_test, best_individual[1]):>10.2f}')

print(f'\nPesos (inv-RMSE): Auto-ETS={w[0]:.3f}, Theta={w[1]:.3f}, SARIMA={w[2]:.3f}')

## Exercicio

### Exercicio 1: Auto-ETS vs Theta vs ARIMA no dataset sintetico

Usando o dataset `ets_synthetic.csv`:

1. Divida em treino (primeiros 156 obs) e teste (ultimos 24 obs)
2. Ajuste Auto-ETS, Theta e ARIMA(1,1,0) nos dados de treino
3. Compare MAE, RMSE e MAPE no conjunto de teste
4. Construa um ensemble com media simples e compare com os modelos individuais
5. O Auto-ETS consegue recuperar o modelo gerador ETS(M,A,M)?

In [ ]:
# Exercicio 1 - Seu codigo aqui
# n_test = 24
# y_train_s = y_synth[:-n_test]
# y_test_s = y_synth[-n_test:]
# result_s = auto_ets(y_train_s, seasonal_period=12)
# ...

## Exercicio

### Exercicio 2: Sensibilidade do criterio de selecao

1. Execute `auto_ets` no airline com `information_criterion='aic'`, `'bic'` e `'aicc'`
2. Os tres criterios selecionam o mesmo modelo?
3. Qual criterio gera previsoes melhores no conjunto de teste?
4. Por que o BIC tende a selecionar modelos mais parcimoniosos?

**Dica:** BIC penaliza mais parametros que o AIC para $n > 8$.

In [ ]:
# Exercicio 2 - Seu codigo aqui
# for crit in ['aic', 'bic', 'aicc']:
#     result = auto_ets(y_train, seasonal_period=12, information_criterion=crit)
#     print(f'{crit}: ETS({result.best_spec})')
# ...